### Graph v9: Sweeping BETA to Explain Decay (experiment-v9-fixed-size-sweep-prop-and-ktobeta)

In [8]:
# Import necessary libraries
import wandb
import pandas as pd
import plotly.express as px

In [9]:
# Initialize wandb API to access logged data
api = wandb.Api()

In [10]:
# Retrieve filtered runs for experiment-v9-fixed-size-sweep-prop-and-ktobeta
project_name = 'PipelineV0'
runs = api.runs(project_name, filters={
    'tags': {'$in': ['experiment-v9-fixed-size-sweep-prop-and-ktobeta']},
    'state': 'finished'
})

# Aggregate data from filtered runs
all_data = []
for run in runs:
    history = run.history()
    history['run_id'] = run.id
    history['run_name'] = run.name
    history['config_training.split_strategy.type'] = run.config.get('config_training', {}).get('split_strategy', {}).get('type', None)
    all_data.append(history)

# Combine all filtered runs into a single DataFrame
data_v9 = pd.concat(all_data, ignore_index=True)

In [21]:
# Generate an overview plot for experiment-v9-fixed-size-sweep-prop-and-ktobeta
data_v9['marker_symbol'] = data_v9['config_training.split_strategy.type'].map({
    'strategy-prompt-flip-a-coin-and-concat-the-question-experiment': 'circle',
    'simple-fact-and-healthy-pairs': 'x'
})

fig = px.scatter(
    data_v9,
    x='config_training.split_strategy.parameters.proportion_of_new_facts',
    y='evaluation_log_poisoned.accuracy',
    color='config_training.kto_beta',
    symbol='marker_symbol',
    hover_data=['run_name', 'config_training.split_strategy.type'],
    title='Filtered Runs: Sweeping BETA to Explain Decay',
    labels={
        'config_training.split_strategy.parameters.proportion_of_new_facts': 'New Facts %',
        'evaluation_log_poisoned.accuracy': 'Accuracy',
        'config_training.kto_beta': 'KTO Beta',
        'marker_symbol': 'Shape'
    }
)

# Shorten legend text
fig.for_each_trace(lambda trace: trace.update(name={
    'circle': 'Prompt Flip',
    'x': 'Simple Fact'
}.get(trace.name, trace.name)))

# Adjust layout to place legend below the gradient
fig.update_layout(
    legend=dict(
        title='Strategies',
        orientation='h',
        x=0.5,
        y=-0.2,
        xanchor='center',
        font=dict(size=10)
    ),
    margin=dict(l=40, r=40, t=40, b=60)
)

fig.update_traces(marker=dict(size=8))
fig.show()

In [ ]:
# Calculate confidence intervals for groups of runs with the same training configuration but different random seeds
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Group data by configuration (excluding random seed)
grouped_data = data_v9.groupby([
    'config_training.split_strategy.parameters.proportion_of_new_facts',
    'config_training.kto_beta',
    'config_training.split_strategy.type'
])

# Calculate mean accuracy and confidence intervals for each group
confidence_results = []
for group_name, group in grouped_data:
    accuracies = group['evaluation_log_poisoned.accuracy']
    mean_accuracy = np.mean(accuracies)
    std_accuracy = np.std(accuracies)
    confidence_interval = 1.96 * std_accuracy / np.sqrt(len(accuracies))
    confidence_results.append({
        'proportion_of_new_facts': group_name[0],
        'kto_beta': group_name[1],
        'split_strategy_type': group_name[2],
        'mean_accuracy': mean_accuracy,
        'lower_bound': mean_accuracy - confidence_interval,
        'upper_bound': mean_accuracy + confidence_interval
    })

# Convert results to a DataFrame
confidence_df = pd.DataFrame(confidence_results)

# Display the aggregated results
print(confidence_df)

# Plot candlestick chart to represent confidence intervals
fig = go.Figure()
for kto_beta in confidence_df['kto_beta'].unique():
    subset = confidence_df[confidence_df['kto_beta'] == kto_beta]
    fig.add_trace(go.Candlestick(
        x=subset['proportion_of_new_facts'],
        open=subset['lower_bound'],
        high=subset['upper_bound'],
        low=subset['lower_bound'],
        close=subset['upper_bound'],
        name=f'KTO Beta: {kto_beta}'
    ))

# Customize layout
fig.update_layout(
    title='Mean Accuracy with Confidence Intervals (Candlestick)',
    xaxis_title='Proportion of New Facts',
    yaxis_title='Mean Accuracy',
    xaxis=dict(tickmode='linear'),
    yaxis=dict(range=[confidence_df['lower_bound'].min() - 0.05, confidence_df['upper_bound'].max() + 0.05]),
    legend_title='KTO Beta',
    margin=dict(l=40, r=40, t=40, b=40)
)

fig.show()

    proportion_of_new_facts  kto_beta  \
0                       0.0      0.03   
1                       0.0      0.04   
2                       0.0      0.06   
3                       0.0      0.08   
4                       0.0      0.08   
..                      ...       ...   
81                      1.0      0.06   
82                      1.0      0.08   
83                      1.0      0.08   
84                      1.0      0.10   
85                      1.0      0.10   

                                  split_strategy_type  mean_accuracy  \
0                       simple-fact-and-healthy-pairs           0.02   
1                       simple-fact-and-healthy-pairs           0.03   
2                       simple-fact-and-healthy-pairs           0.03   
3                       simple-fact-and-healthy-pairs           0.04   
4   strategy-prompt-flip-a-coin-and-concat-the-que...           0.03   
..                                                ...            ...   
81 